In [ ]:
import os
import pandas as pd
import shutil
import ast

In [ ]:
CSV_PATH       = 'E:/TFM/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv'
BASE_IMAGE_DIR = 'E:/TFM/PADCHEST'
OUTPUT_DIR     = 'E:/TFM/Dataset_fewshot_15clases'

IMAGES_PER_CLASS = 100
RANDOM_STATE     = 42
GROUP_COL        = 'PatientID'

TRAIN_CLASSES = [
    'cardiomegaly',
    'aortic elongation',
    'scoliosis',
    'fibrotic band',
    'costophrenic angle blunting',
    'infiltrates',
    'nodule',
    'calcified granuloma',
    'diaphragmatic eventration',
    'gynecomastia',
    'aortic atheromatosis',
    'apical pleural thickening',
    'vascular hilar enlargement',
    'laminar atelectasis',
    'interstitial pattern',
    'hiatal hernia',
    'callus rib fracture',
    'hemidiaphragm elevation',
    'alveolar pattern',
    'vertebral anterior compression',
]

TEST_CLASSES = [
    'COPD signs',
    'pneumonia',
    'bronchiectasis',
    'pleural effusion',
    'kyphosis',
]

ALL_CLASSES = TRAIN_CLASSES + TEST_CLASSES

In [3]:
df = pd.read_csv(CSV_PATH, low_memory=False)
df = df[df['Labels'].notna()].copy()
df['Labels'] = df['Labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df = df[df['Pediatric'] != 'Yes'].copy()
df = df[(df['Projection'] == 'PA') & (df['ViewPosition_DICOM'] == 'PA')].copy()
df = df.drop_duplicates(subset=['ImageID']).copy()

label_counts = df.explode('Labels')['Labels'].value_counts()

def assign_rarest_label(labels):
    if not isinstance(labels, list) or len(labels) == 0:
        return None
    candidates = [l for l in labels if l in ALL_CLASSES]
    if not candidates:
        return None
    return min(candidates, key=lambda l: label_counts.get(l, 0))

df['target_label'] = df['Labels'].apply(assign_rarest_label)
df = df[df['target_label'].notna()].copy()
df = df[df['target_label'].isin(ALL_CLASSES)].copy()

print('Imágenes disponibles por clase:')
print(df['target_label'].value_counts())

Imágenes disponibles por clase:
target_label
scoliosis                         353
COPD signs                        348
aortic elongation                 295
vascular hilar enlargement        252
aortic atheromatosis              247
cardiomegaly                      241
fibrotic band                     207
apical pleural thickening         199
laminar atelectasis               174
costophrenic angle blunting       152
alveolar pattern                  127
calcified granuloma               127
hemidiaphragm elevation           127
nodule                            125
callus rib fracture               125
pleural effusion                  122
infiltrates                       122
diaphragmatic eventration         117
vertebral anterior compression    115
hiatal hernia                     114
bronchiectasis                    112
interstitial pattern              111
gynecomastia                      108
kyphosis                           80
pneumonia                          55
Name:

In [ ]:
def build_path(row):
    return os.path.join(BASE_IMAGE_DIR, str(row['ImageDir']), row['ImageID'])

df['image_path'] = df.apply(build_path, axis=1)

splits = []
for label, g in df.groupby('target_label'):
    target_n = min(IMAGES_PER_CLASS, len(g))
    g_sampled = g.sample(n=target_n, random_state=RANDOM_STATE).copy()
    g_sampled['split'] = 'train' if label in TRAIN_CLASSES else 'test'
    splits.append(g_sampled)

final_df = pd.concat(splits).reset_index(drop=True)

print('\nTotal final:', len(final_df))
print('\nImágenes por clase y split:')
print(pd.crosstab(final_df['target_label'], final_df['split']))


Total final: 1250

Imágenes por clase y split:
split                           test  train
target_label                               
COPD signs                        50      0
alveolar pattern                   0     50
aortic atheromatosis               0     50
aortic elongation                  0     50
apical pleural thickening          0     50
bronchiectasis                    50      0
calcified granuloma                0     50
callus rib fracture                0     50
cardiomegaly                       0     50
costophrenic angle blunting        0     50
diaphragmatic eventration          0     50
fibrotic band                      0     50
gynecomastia                       0     50
hemidiaphragm elevation            0     50
hiatal hernia                      0     50
infiltrates                        0     50
interstitial pattern               0     50
kyphosis                          50      0
laminar atelectasis                0     50
nodule                      

In [ ]:
for _, row in final_df.iterrows():
    label = row['target_label'].replace(' ', '_')
    dst_dir = os.path.join(OUTPUT_DIR, row['split'], label)
    os.makedirs(dst_dir, exist_ok=True)
    src = row['image_path']
    dst = os.path.join(dst_dir, os.path.basename(src))
    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        print(f'⚠ No encontrada: {src}')

print('\n✅ Dataset creado.')
assert final_df['ImageID'].is_unique, '❌ Hay ImageIDs duplicados'

train_classes = set(final_df[final_df['split'] == 'train']['target_label'].unique())
test_classes  = set(final_df[final_df['split'] == 'test']['target_label'].unique())
assert train_classes.isdisjoint(test_classes), '❌ Clases compartidas entre train y test'
print('✅ Sin solapamiento de clases entre train y test')

final_df[['ImageID', 'target_label', 'split', GROUP_COL, 'image_path']].to_csv(
    os.path.join(OUTPUT_DIR, 'dataset_summary.csv'), index=False
)


✅ Dataset creado.
✅ Sin solapamiento de clases entre train y test
